# O-RAG — RAGAS Evaluation (100% Local, No API Key)

Evaluates the offline RAG pipeline using **RAGAS** with the **same Qwen + Nomic servers** that the app already runs — no OpenAI key required.

```
Judge LLM   →  Qwen llama-server  (port 8080, /v1/chat/completions)
Embeddings  →  Nomic llama-server (port 8081, /v1/embeddings)
```

| Metric | Type | What it measures |
|---|---|---|
| **Faithfulness** | LLM-judge | Answer is grounded in retrieved context |
| **Answer Relevancy** | Embedding | Answer addresses the question |
| **Context Recall** | LLM-judge | Retrieved context covers the ground-truth |
| **Context Precision** | LLM-judge | Retrieved chunks are relevant (low noise) |
| **Answer Similarity** | Embedding | Semantic closeness to ground-truth |

> **Note:** LLM-judge metrics require the model to emit valid JSON.  
> Small Qwen 0.5B may occasionally fail — the notebook handles this gracefully  
> and always computes the embedding-only metrics as a reliable fallback.

**Install once:**
```bash
pip install ragas datasets langchain-openai sentence-transformers pandas matplotlib
```

## 0 · Configuration

In [ ]:
import os, sys
from pathlib import Path

# ── Local server ports (must match config.py) ─────────────────────────
QWEN_PORT  = 8080   # generation server  →  /v1/chat/completions
NOMIC_PORT = 8081   # embedding server   →  /v1/embeddings

# ── Qwen model name as reported by llama-server  ─────────────────────
# llama-server accepts any non-empty string here; we just label it.
QWEN_MODEL_NAME  = "qwen"
NOMIC_MODEL_NAME = "nomic-embed-text-v1.5"

# ── Model file paths (only needed when NO_BOOT = False) ───────────────
QWEN_MODEL_PATH  = r"D:\Work\8th_Sem\Models\Qwen3.5-2B-Q4_K_M.gguf"   
NOMIC_MODEL_PATH = r"D:\Work\8th_Sem\Models\nomic-embed-text-v1.5.Q8_0.gguf"   

# ── Set True if servers are already running ───────────────────────────
NO_BOOT = False

# ── Retrieval settings ────────────────────────────────────────────────
TOP_K = 4

# ── Dataset / output paths ────────────────────────────────────────────
DATASET_PATH = Path("sample_dataset.json")
RESULTS_DIR  = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Add Python backend to sys.path ────────────────────────────────────
_PYTHON_SRC = (
    Path(".").resolve().parent
    / "orag" / "android" / "app" / "src" / "main" / "python"
)
if str(_PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(_PYTHON_SRC))

# Suppress noisy tokenizer warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print(f"Python src: {_PYTHON_SRC}")
print("Configuration loaded.")

Python src: D:\Work\8th_Sem\ORagFlutter\orag\android\app\src\main\python
Configuration loaded.


## 1 · Start Servers (skip if already running)

In [21]:
# One variable to set — path to the llama-server.exe binary
# Run this when you're done to free memory
from server_utils import stop_servers
stop_servers(_server_procs)

LLAMA_SERVER_BIN = str(
    Path(".").resolve().parent / "orag" / "llamacpp_bin" / "llama-server.exe"
)

from server_utils import launch_servers
_server_procs = launch_servers(
    no_boot     = NO_BOOT,
    qwen_port   = QWEN_PORT,
    nomic_port  = NOMIC_PORT,
    qwen_model  = QWEN_MODEL_PATH,
    nomic_model = NOMIC_MODEL_PATH,
    binary      = LLAMA_SERVER_BIN,
)


NameError: name '_server_procs' is not defined

## 2 · Build RAGAS Judge LLM & Embeddings (Local)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ── Judge LLM: Qwen via llama-server OpenAI-compatible endpoint ───────
# llama-server exposes /v1/chat/completions on the same port as /completion
_qwen_base = f"http://127.0.0.1:{QWEN_PORT}/v1"

judge_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=QWEN_MODEL_NAME,
        openai_api_key="local",        # placeholder — not validated locally
        openai_api_base=_qwen_base,
        temperature=0,
        max_tokens=512,
        request_timeout=60,
    )
)

# ── Embeddings: Nomic via llama-server OpenAI-compatible endpoint ─────
_nomic_base = f"http://127.0.0.1:{NOMIC_PORT}/v1"

judge_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(
        model=NOMIC_MODEL_NAME,
        openai_api_key="local",
        openai_api_base=_nomic_base,
        check_embedding_ctx_length=False,  # Nomic uses its own ctx management
    )
)

print(f"Judge LLM  → {_qwen_base}")
print(f"Embeddings → {_nomic_base}")

# ── Quick smoke-test ─────────────────────────────────────────────────
try:
    test_resp = judge_llm.langchain_llm.invoke("Reply with: OK")
    print(f"LLM smoke-test: {test_resp.content[:60]}")
except Exception as e:
    print(f"LLM smoke-test FAILED: {e}")
    print("Check that the Qwen server is running and that /v1/chat/completions is supported.")

Judge LLM  → http://127.0.0.1:8080/v1
Embeddings → http://127.0.0.1:8081/v1


C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\2887235392.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(
C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\2887235392.py:23: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(


LLM smoke-test FAILED: Connection error.
Check that the Qwen server is running and that /v1/chat/completions is supported.


## 3 · Initialize Pipeline

In [ ]:
from storage import init_db
import pipeline as pipe_mod

init_db()
pipe_mod.retriever.reload()
print("Pipeline ready. Docs in DB:", len(pipe_mod.list_documents()))

[storage] Migrated: added parent_chunk_idx to chunks
[storage] Migrated: added embedding column to chunks
Pipeline ready. Docs in DB: 0


## 4 · Load Dataset

In [ ]:
import json, pandas as pd

with open(DATASET_PATH, encoding="utf-8") as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data)} QA pairs.")
pd.DataFrame(eval_data)[["question","ground_truth"]].head(10)

Loaded 5 QA pairs.


,question,ground_truth
0,What is retrieval-augmented generation?,Retrieval-Augmented Generation (RAG) is a tech...
1,How does BM25 differ from dense retrieval?,"BM25 is a sparse, term-frequency-based retriev..."
2,What is Reciprocal Rank Fusion?,Reciprocal Rank Fusion (RRF) is a rank-based f...
3,Explain small-to-big chunk retrieval.,Small-to-big chunk retrieval is a hierarchical...
4,What is the role of a reranker in a RAG pipeline?,A reranker re-scores an initial set of retriev...


## 5 · Ingest Documents (if any)

In [ ]:
from storage import list_documents as _list_docs

already = {d["name"] for d in _list_docs()}
for item in eval_data:
    for doc_path in item.get("doc_paths", []):
        name = Path(doc_path).name
        if name in already:
            print(f"  skip  : {name}"); continue
        if not Path(doc_path).is_file():
            print(f"  WARN  : not found — {doc_path}"); continue
        ok, msg = pipe_mod.ingest_document(doc_path)
        print(f"  {'OK' if ok else 'FAIL'}: {msg}")
        already.add(name)

print(f"\nTotal docs in DB: {len(_list_docs())}")


Total docs in DB: 0


## 6 · Run RAG Queries & Collect Results

In [ ]:
collected = []

for i, item in enumerate(eval_data):
    question      = item["question"]
    ground_truth  = item["ground_truth"]
    pre_contexts  = item.get("contexts")   # optional pre-supplied contexts

    print(f"\n[{i+1}/{len(eval_data)}] {question[:80]}")

    if pre_contexts is not None:
        # Offline mode — use pre-supplied contexts, generate answer via LLM
        contexts = pre_contexts
        _, answer = pipe_mod.chat_direct(question=question)
        print(f"  answer (pre-ctx): {answer[:120]}")
    else:
        # Live mode — full hybrid retrieval + generation
        ok, answer, _ = pipe_mod.ask(question)
        if not ok:
            answer = f"ERROR: {answer}"
        results = pipe_mod.retriever.query_with_expansion(question, top_k=TOP_K)
        if not results:
            results = pipe_mod.retriever.query(question, top_k=TOP_K)
        contexts = [t for t, _, _ in results]
        print(f"  answer          : {answer[:120]}")
        print(f"  contexts fetched: {len(contexts)}")

    collected.append({
        "question":     question,
        "ground_truth": ground_truth,
        "answer":       answer,
        "contexts":     contexts if contexts else [""],
    })

print(f"\n✓ Collected {len(collected)} records.")


[1/5] What is retrieval-augmented generation?
  answer (pre-ctx): No LLM model loaded. Please load a GGUF model first.

[2/5] How does BM25 differ from dense retrieval?
  answer (pre-ctx): No LLM model loaded. Please load a GGUF model first.

[3/5] What is Reciprocal Rank Fusion?
  answer (pre-ctx): No LLM model loaded. Please load a GGUF model first.

[4/5] Explain small-to-big chunk retrieval.
  answer (pre-ctx): No LLM model loaded. Please load a GGUF model first.

[5/5] What is the role of a reranker in a RAG pipeline?
  answer (pre-ctx): No LLM model loaded. Please load a GGUF model first.

✓ Collected 5 records.


## 7 · RAGAS Evaluation

**Strategy:**
- Primary: all 5 metrics using the local Qwen judge + Nomic embeddings.
- If the local Qwen is too small to produce valid JSON for LLM-judge metrics,  
  `raise_exceptions=False` makes RAGAS skip those entries instead of crashing.
- `answer_similarity` is embedding-only and always works regardless of model size.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_similarity,   # embedding-only — always reliable
)

ragas_ds = Dataset.from_dict({
    "question":     [r["question"]     for r in collected],
    "answer":       [r["answer"]       for r in collected],
    "contexts":     [r["contexts"]     for r in collected],
    "ground_truth": [r["ground_truth"] for r in collected],
})

print("Running RAGAS evaluation with local Qwen + Nomic…")
result = evaluate(
    dataset=ragas_ds,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision,
        answer_similarity,
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
    raise_exceptions=False,   # don't crash on small-model JSON failures
)

print("\nDone!")
print(result)

Running RAGAS evaluation with local Qwen + Nomic…


C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\2310041072.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\2310041072.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\2310041072.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\SJABI\AppData\Local\Temp\ipykernel_4408\231004


Done!
{'faithfulness': nan, 'answer_relevancy': nan, 'context_recall': nan, 'context_precision': nan, 'answer_similarity': nan}


## 7b · Fallback — Embedding-Only Metrics (No LLM Required)

Run this cell **instead of cell 7** if the Qwen server is not running  
or if you want fast, fully-offline scoring using only sentence-transformers.

In [ ]:
# ── Embedding-only metrics via sentence-transformers ──────────────────
# Uses the same Nomic model family locally through SBERT.
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util
import numpy as np

# Use a small but capable model (downloads once, ~90 MB)
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
sbert = SentenceTransformer(SBERT_MODEL)
print(f"Loaded SBERT model: {SBERT_MODEL}")

def _cosine(a, b):
    return float(util.cos_sim(a, b)[0][0])

fallback_rows = []
for rec in collected:
    q_emb  = sbert.encode(rec["question"],     convert_to_tensor=True)
    a_emb  = sbert.encode(rec["answer"],       convert_to_tensor=True)
    gt_emb = sbert.encode(rec["ground_truth"], convert_to_tensor=True)

    # answer_relevancy proxy: cosine(question, answer)
    ans_rel = _cosine(q_emb, a_emb)

    # answer_similarity proxy: cosine(answer, ground_truth)
    ans_sim = _cosine(a_emb, gt_emb)

    # context_relevance proxy: max cosine(question, context_i)
    if rec["contexts"] and rec["contexts"] != [""]:
        ctx_embs = sbert.encode(rec["contexts"], convert_to_tensor=True)
        ctx_sims = [_cosine(q_emb, c) for c in ctx_embs]
        ctx_rel  = float(np.mean(ctx_sims))
    else:
        ctx_rel = 0.0

    fallback_rows.append({
        "question":         rec["question"],
        "answer_relevancy": round(ans_rel, 4),
        "answer_similarity": round(ans_sim, 4),
        "context_relevance": round(ctx_rel, 4),
    })

fb_df = pd.DataFrame(fallback_rows)
print("\nFallback embedding-only scores:")
display(fb_df)
print("\nMeans:")
display(fb_df[["answer_relevancy","answer_similarity","context_relevance"]].mean().round(4))

KeyboardInterrupt: 

## 8 · Results Summary

In [ ]:
METRICS = [
    "faithfulness",
    "answer_relevancy",
    "context_recall",
    "context_precision",
    "answer_similarity",
]

scores_df = result.to_pandas()

# Per-question table
display(scores_df[["question", *[m for m in METRICS if m in scores_df.columns]]].round(4))

# Aggregate summary
avail = [m for m in METRICS if m in scores_df.columns]
summary = scores_df[avail].agg(["mean", "min", "max"]).T
summary.columns = ["Mean", "Min", "Max"]
print("\n=== Aggregate Scores ===")
display(summary.round(4))

## 9 · Save CSV Report

In [ ]:
import csv, datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = RESULTS_DIR / f"{ts}_ragas.csv"

avail_metrics = [m for m in METRICS if m in scores_df.columns]
fieldnames = ["question", "ground_truth", "answer", "num_contexts", *avail_metrics]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for i, rec in enumerate(collected):
        row = {
            "question":     rec["question"],
            "ground_truth": rec["ground_truth"],
            "answer":       rec["answer"],
            "num_contexts": len(rec["contexts"]),
        }
        if i < len(scores_df):
            for m in avail_metrics:
                row[m] = round(float(scores_df[m].iloc[i]), 4)
        writer.writerow(row)

print(f"Report saved → {csv_path}")

## 10 · Visualise Scores

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

avail = [m for m in METRICS if m in scores_df.columns]
means = scores_df[avail].mean()

colors = ["#4C72B0","#55A868","#C44E52","#8172B2","#CCB974"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle("O-RAG · RAGAS Evaluation Results", fontsize=13, fontweight="bold")

# Bar chart
bars = axes[0].bar(means.index, means.values,
                   color=colors[:len(means)], width=0.5, edgecolor="white")
axes[0].set_ylim(0, 1.12)
axes[0].set_ylabel("Score")
axes[0].set_title("Mean Scores")
axes[0].set_xticklabels(means.index, rotation=22, ha="right")
for bar, v in zip(bars, means.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.02,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=9)

# Radar chart
N = len(avail)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
vals   = means.values.tolist() + [means.values[0]]
ax2 = plt.subplot(122, polar=True)
ax2.plot(angles, vals, "o-", linewidth=2, color="#4C72B0")
ax2.fill(angles, vals, alpha=0.2, color="#4C72B0")
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels([m.replace("_","\n") for m in avail], fontsize=8)
ax2.set_ylim(0, 1)
ax2.set_title("Radar", pad=14)

plt.tight_layout()
plot_path = RESULTS_DIR / f"{ts}_ragas_plot.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {plot_path}")